In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
from tablevault import tablevault
import os
vault = tablevault.Vault(user_id="jinjin",
                            process_name="semantic_verbalizers_zero_shot_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

classifier = pipeline(
    "zero-shot-classification",
    model=model_name,
    device=device,
)

print(model_name)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
texts = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print(texts[0])


'The read operation timed out' thrown while requesting HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/.huggingface.yaml
Retrying in 1s [Retry 1/5].


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [4]:
candidate_labels = ["same meaning", "different meaning"]
hypothesis_template = "The two sentences have {}."

batch_size = 32
raw_outputs = []
y_pred = []
score_dicts = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i + batch_size]
    batch_outputs = classifier(
        batch_texts,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
        batch_size=batch_size,
        truncation=True,
        max_length=256,
    )
    if isinstance(batch_outputs, dict):
        batch_outputs = [batch_outputs]

    for out in batch_outputs:
        raw_outputs.append(out)
        scores = {label: float(score) for label, score in zip(out["labels"], out["scores"])}
        same_score = scores.get("same meaning", 0.0)
        diff_score = scores.get("different meaning", 0.0)
        score_dicts.append({
            "same meaning": same_score,
            "different meaning": diff_score,
        })
        y_pred.append(1 if out["labels"][0] == "same meaning" else 0)

y_pred = np.array(y_pred)
print("done")
print(raw_outputs[0])


  0%|          | 0/13 [00:00<?, ?it/s]

done
{'sequence': 'Sentence 1: He said the foodservice pie business doesn \'t fit the company \'s long-term growth strategy .\nSentence 2: " The foodservice pie business does not fit our long-term growth strategy .', 'labels': ['different meaning', 'same meaning'], 'scores': [0.9994259476661682, 0.000574022124055773]}


In [ ]:

vault.create_record_list("distilbert-paraphrase-meaning-prediction", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("distilbert-paraphrase-meaning-prediction", 
                        {
                            "prediction": y_pred[i],
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "This dataset stores example-level paraphrase predictions produced by a zero-shot DistilBERT MNLI classifier on the GLUE MRPC validation set. Each record corresponds to one sentence-pair example from glue_mrpc_validation and contains a single field, prediction, where 1 indicates the model predicted the two sentences have the same meaning (paraphrase) and 0 indicates different meaning (not paraphrase). In this workflow, the dataset serves as the model output table used to persist prediction results, link them back to the source MRPC records, and support downstream evaluation and summary metrics such as accuracy, F1, and the classification report."
embedding = get_embeddings(description)
vault.create_description("distilbert-paraphrase-meaning-prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions", "model": "typeform/distilbert-base-uncased-mnli", "inference_type": "zero-shot classification", "verbalizers": "same meaning,different meaning", "source": "glue/mrpc", "split": "validation", "size": "408", "input_type": "sentence pair", "domain": "news", "output": "binary paraphrase prediction"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-paraphrase-meaning-prediction", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["different meaning", "same meaning"]))


{'accuracy': 0.4117647058823529, 'f1': 0.28994082840236685}
                   precision    recall  f1-score   support

different meaning       0.34      0.92      0.50       129
     same meaning       0.83      0.18      0.29       279

         accuracy                           0.41       408
        macro avg       0.59      0.55      0.39       408
     weighted avg       0.68      0.41      0.36       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("same meaning score:", round(score_dicts[i]["same meaning"], 6))
    print("different meaning score:", round(score_dicts[i]["different meaning"], 6))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("same meaning score:", round(score_dicts[i]["same meaning"], 6))
    print("different meaning score:", round(score_dicts[i]["different meaning"], 6))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0
same meaning score: 0.000574
different meaning score: 0.999426
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0
same meaning score: 0.341497
different meaning score: 0.658503
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0
same meaning score: 0.47575
different meaning score: 0.52425
sentence1: The AFL-CIO is waiting until O

In [7]:
vault.create_record_list("semantic_verbalizers_zero_shot_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("semantic_verbalizers_zero_shot_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert-paraphrase-meaning-prediction": [0, len(ds)]
                    })

summary

description = "semantic_verbalizers_zero_shot_mrpc_summary is a summary dataset for the zero-shot MRPC evaluation run in this notebook. It contains aggregate performance results for predictions generated on the glue_mrpc_validation dataset using the typeform/distilbert-base-uncased-mnli zero-shot classifier with candidate labels \u201csame meaning\u201d and \u201cdifferent meaning.\u201d The dataset is stored as a record list with three fields: accuracy (float), f1 (float), and classification_report (string). In this workflow, it serves as the experiment-level evaluation artifact, linking the full validation set and the corresponding prediction dataset (distilbert-paraphrase-meaning-prediction) to a compact summary of model performance for tracking, comparison, and retrieval."
embedding = get_embeddings(description)
vault.create_description("semantic_verbalizers_zero_shot_mrpc_summary", description, embedding)

properties = {"dataset_type": "evaluation summary", "task": "paraphrase detection", "benchmark": "GLUE MRPC", "source": "glue_mrpc_validation", "split": "validation", "model": "typeform/distilbert-base-uncased-mnli", "inference_method": "zero-shot classification", "verbalizer_labels": "same meaning|different meaning", "input_format": "sentence pair", "metrics": "accuracy,f1,classification_report", "record_count": "1"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("semantic_verbalizers_zero_shot_mrpc_summary", cat, embedding, prop)


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.4117647058823529,
 'f1': 0.28994082840236685}

In [ ]:
description = "This notebook evaluates zero-shot paraphrase detection on the GLUE MRPC validation set using the Hugging Face model typeform/distilbert-base-uncased-mnli. It frames each sentence pair as a zero-shot classification problem with the candidate labels \u201csame meaning\u201d and \u201cdifferent meaning,\u201d runs batched inference, and converts the top predicted label into a binary paraphrase prediction. The workflow loads MRPC examples from TableVault, computes predictions and label scores, measures performance with accuracy, F1, and a classification report, and inspects example successes and errors. It then stores per-example predictions, aggregate evaluation results, and notebook/process metadata back into TableVault, using OpenAI text embeddings to attach searchable descriptions and properties." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("semantic_verbalizers_zero_shot_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "approach": "zero-shot classification with semantic verbalizers", "model": "typeform/distilbert-base-uncased-mnli", "dataset": "GLUE/MRPC validation", "input_format": "sentence pair", "labels": "same meaning, different meaning", "frameworks": "PyTorch, Hugging Face Transformers, Datasets", "evaluation": "accuracy, f1-score, classification report", "output_artifacts": "per-example predictions and summary metrics in TableVault", "tracking": "TableVault metadata and OpenAI embeddings"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("semantic_verbalizers_zero_shot_mrpc", cat, embedding, prop)